In [25]:
import numpy as np
import pandas as pd
import re
from spellchecker import SpellChecker
import json ,csv
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')




In [ ]:
df =  pd.read_csv("..\\data\\processed\\dataset.csv")

In [27]:
df.head()

,id,Contains,SideEffect,Chemical_Class,Habit_Forming,Therapeutic_Class
0,1,Haloperidol (0.5mg),Most side effects do not require any medical a...,Butyrophenone Derivative,No,NEURO CNS
1,2,Bevacizumab (100mg),Most side effects do not require any medical a...,Monoclonal antibody (mAb),No,ANTI NEOPLASTICS
2,3,Darbepoetin alfa (40mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED
3,4,Darbepoetin alfa (25mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED
4,5,Darbepoetin alfa (60mcg),Most side effects do not require any medical a...,"Amino Acids, Peptides Analogues",No,BLOOD RELATED


In [28]:
df.isna().sum()

id                       0
Contains                 0
SideEffect               0
Chemical_Class       91334
Habit_Forming            0
Therapeutic_Class        0
dtype: int64

In [29]:
df.columns

Index(['id', 'Contains', 'SideEffect', 'Chemical_Class', 'Habit_Forming',
       'Therapeutic_Class'],
      dtype='object')

In [30]:
ing = df[["id","Contains"]]

In [31]:
def extract_special(s):
    return ''.join(re.findall(r'[^a-zA-Z0-9\s]', str(s)))


In [32]:
ing["special_chars"] = ing["Contains"].apply(extract_special)

In [33]:
ing.tail()

,id,Contains,special_chars
192802,192803,Cefpodoxime Proxetil (200mg),()
192803,192804,Gentamicin (0.3% w/v),(.%/)
192804,192805,Oxcarbazepine (900mg),()
192805,192806,Telmisartan (40mg)+ Hydrochlorothiazide (12.5mg),()+(.)
192806,192807,Pregabalin (50mg)+ Duloxetine (20mg),()+()


In [34]:
def clean(row):
    original = str(row["Contains"])
    specials = str(row["special_chars"])

    
    if "(" in specials or ")" in specials or "+" in specials:
        original = re.sub(r"\(.*?\)", " ", original)  
        original = original.replace("+", "##")        
        original = original.replace(",", " ")         
        original = re.sub(r"\s+", " ", original).strip()

    return original

In [35]:
ing["ingredients"] = ing.apply(clean, axis=1)

In [36]:
ing.tail()

,id,Contains,special_chars,ingredients
192802,192803,Cefpodoxime Proxetil (200mg),(),Cefpodoxime Proxetil
192803,192804,Gentamicin (0.3% w/v),(.%/),Gentamicin
192804,192805,Oxcarbazepine (900mg),(),Oxcarbazepine
192805,192806,Telmisartan (40mg)+ Hydrochlorothiazide (12.5mg),()+(.),Telmisartan ## Hydrochlorothiazide
192806,192807,Pregabalin (50mg)+ Duloxetine (20mg),()+(),Pregabalin ## Duloxetine


In [ ]:
with open('..\\data\\processed\\ingredients_dictionary.json', 'r', encoding='utf-8') as f:
    ingredient_dict = json.load(f)

In [38]:
def map_cell(cell):
    if pd.isna(cell) or str(cell).strip() == "":
        return "Missing data"
    
    terms = [t.strip() for t in str(cell).split('##') if t.strip()]
    mapped_terms = [ingredient_dict.get(t, t) for t in terms]
    unique_mapped = list(dict.fromkeys(mapped_terms))
    return " ## ".join(unique_mapped)

In [39]:
ing['ingredients_mapped'] = ing['ingredients'].apply(map_cell)

In [40]:
ing.head()

,id,Contains,special_chars,ingredients,ingredients_mapped
0,1,Haloperidol (0.5mg),(.),Haloperidol,Haloperidol
1,2,Bevacizumab (100mg),(),Bevacizumab,Bevacizumab
2,3,Darbepoetin alfa (40mcg),(),Darbepoetin alfa,Darbepoetin alfa
3,4,Darbepoetin alfa (25mcg),(),Darbepoetin alfa,Darbepoetin alfa
4,5,Darbepoetin alfa (60mcg),(),Darbepoetin alfa,Darbepoetin alfa


In [41]:

ingredients = ing[["id","ingredients_mapped"]]

In [42]:
ingredients.tail()

,id,ingredients_mapped
192802,192803,Cefpodoxime
192803,192804,Gentamicin
192804,192805,Oxcarbazepine
192805,192806,Telmisartan ## Hydrochlorothiazide
192806,192807,Pregabalin ## Duloxetine


In [ ]:
ingredients.to_csv("..\\data\\processed\\ingredients.csv",index=False)